# Qwen2.5-3B-Instruct GPTQ-Int4 — vLLM Kaggle T4x2 Serving
## vLLM · OpenAI-compatible API · cloudflared public tunnel · optional API key

| Setting | Value |
|---------|-------|
| Model | `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` |
| Engine | [vLLM](https://github.com/vllm-project/vllm) — continuous batching, PagedAttention |
| Quantization | GPTQ-Int4 (~2 GB vs 6 GB float16) |
| Concurrency | High (64+) — vLLM batches requests internally |
| Endpoint | OpenAI `POST /v1/chat/completions` |
| Tunnel | cloudflare quick tunnel — no account needed |
| Auth | Optional `Authorization: Bearer <key>` |

> vLLM is the correct engine for concurrent serving. It replaces the
> FastAPI + `model.generate()` approach, which serialises on the GPU and
> cannot handle concurrent requests without CUDA conflicts.


In [ ]:
# vLLM includes its own OpenAI-compatible server — no custom FastAPI needed.
!pip install -q vllm


In [ ]:
# ── All tunable parameters — edit here, nowhere else ─────────────────────────

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4"

PORT = 8000

# Leave "" for open access; set a string to require Bearer auth.
API_KEY = ""   # e.g. "my-secret-42"

# Inference defaults (vLLM also accepts these per-request in the JSON body)
MAX_NEW_TOKENS_DEFAULT  = 2048
TEMPERATURE_DEFAULT     = 0.7
TOP_P_DEFAULT           = 0.9

# vLLM engine settings
TENSOR_PARALLEL_SIZE = 1        # 3B fits on 1 T4 (16 GB); set to 2 for 7B+
MAX_MODEL_LEN        = 4096     # max sequence length (prompt + output)
GPU_MEMORY_UTILISATION = 0.90   # fraction of VRAM vLLM may use


In [ ]:
import os, re, time, subprocess, threading
import torch
from datetime import datetime

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
n_gpu = torch.cuda.device_count()
print(f"GPUs     : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB")
print(f"model    : {MODEL_NAME}")
print("=" * 60)

if n_gpu == 0:
    raise RuntimeError("No GPU found — enable GPU accelerator in Kaggle settings.")


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login
    login(token=_tok, add_to_git_credential=False)
    print("HuggingFace: authenticated via Kaggle Secret HF_TOKEN")
except Exception as _e:
    print(f"HuggingFace: no token ({_e}) — continuing unauthenticated")


In [ ]:
# vLLM's built-in OpenAI server handles:
#   - Continuous batching (multiple concurrent requests batched on the GPU)
#   - PagedAttention (efficient KV cache)
#   - /v1/models, /v1/chat/completions, /v1/completions out of the box
#   - tensor parallelism across both T4s via --tensor-parallel-size 2

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model",                  MODEL_NAME,
    "--tensor-parallel-size",   str(TENSOR_PARALLEL_SIZE),
    "--max-model-len",          str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILISATION),
    "--port",                   str(PORT),
    "--host",                   "0.0.0.0",
    "--quantization", "gptq", "--dtype", "float16",
]
if API_KEY:
    vllm_cmd += ["--api-key", API_KEY]

print("Starting vLLM server (model download + engine init ~3-5 min first run) ...")
print(f"Command: {' '.join(vllm_cmd)}")
print()

vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

# Stream vLLM output and wait until the server is ready
import select, sys

def _tail_vllm():
    for line in vllm_proc.stdout:
        print(f"[vllm] {line}", end="")
        if "Application startup complete" in line or "Uvicorn running" in line:
            print("\n✅ vLLM server is ready.")

_tail_thread = threading.Thread(target=_tail_vllm, daemon=True)
_tail_thread.start()

# Poll /health until the server responds (max 5 min)
import urllib.request, json as _json

deadline = time.time() + 300
while time.time() < deadline:
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as r:
            if r.status == 200:
                print("Server responding on localhost.")
                break
    except Exception:
        pass
else:
    raise RuntimeError("vLLM server did not start within 5 minutes — check output above.")

# ── cloudflared quick tunnel ──────────────────────────────────────────────────
CF_BIN = "/tmp/cloudflared"
if not os.path.exists(CF_BIN):
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q", "-O", CF_BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    subprocess.run(["chmod", "+x", CF_BIN], check=True)

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
)

public_url = None
deadline = time.time() + 45
while time.time() < deadline:
    line = cf_proc.stderr.readline()
    if not line:
        time.sleep(0.2); continue
    hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", line)
    if hit:
        public_url = hit.group(0); break

if not public_url:
    try:
        remaining = cf_proc.stderr.read(2000)
        hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", remaining)
        if hit:
            public_url = hit.group(0)
    except Exception:
        pass

if public_url:
    print()
    print("=" * 60)
    print("PUBLIC URL:")
    print(f"  {public_url}")
    print()
    print(f"  GET  {public_url}/health")
    print(f"  GET  {public_url}/v1/models")
    print(f"  POST {public_url}/v1/chat/completions")
    if API_KEY:
        print(f"\n  Authorization: Bearer {API_KEY}")
    else:
        print("\n  No auth required")
    print("=" * 60)
else:
    print("WARNING: cloudflare URL not found — try: cf_proc.stderr.read(2000)")


In [ ]:
import urllib.request, json

url   = f"http://localhost:{PORT}/v1/chat/completions"
hdrs  = {"Content-Type": "application/json"}
if API_KEY:
    hdrs["Authorization"] = f"Bearer {API_KEY}"

payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is 2 + 2? Answer in one word."},
    ],
    "max_tokens": 20,
    "temperature": 0.0,
}).encode()

req  = urllib.request.Request(url, data=payload, headers=hdrs, method="POST")
with urllib.request.urlopen(req, timeout=60) as r:
    data = json.loads(r.read())

print("Answer :", data["choices"][0]["message"]["content"])
print("Usage  :", data["usage"])


In [ ]:
# ==============================================================================
# ⚙️  STRESS TEST CONFIGURATION
# ==============================================================================

API_TYPE = "openai"

# Automatically use the cloudflared URL set by the server cell.
# If you run this cell before the server cell, paste the URL manually.
try:
    API_URL = f"{public_url}/v1/chat/completions"
except NameError:
    API_URL = "https://YOUR-TUNNEL.trycloudflare.com/v1/chat/completions"

# Re-uses API_KEY and MODEL_NAME from Cell 2 — no need to set them again.
MODEL_NAME_TEST = MODEL_NAME

TOTAL_REQUESTS    = 1000
CONCURRENCY_LIMIT = 64    # vLLM handles high concurrency via continuous batching
MAX_TOKENS        = 2048
MAX_RETRIES       = 3

PROMPTS = [
    "Explain quantum entanglement to a 5-year-old.",
    "Write a haiku about a server crashing.",
    "List 5 fun facts about dolphins.",
    "Translate 'Hello world' into Python code.",
    "What is the capital of Australia?",
    "Summarise the plot of Romeo and Juliet in one sentence.",
    "Why is the sky blue?",
    "Write a short email declining a wedding invitation politely.",
    "Explain the difference between TCP and UDP.",
    "Give me a recipe for pancakes.",
]

# ==============================================================================
# 🚀  ENGINE
# ==============================================================================

import asyncio, aiohttp, time, random, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class StressTester:
    def __init__(self):
        self.results    = []
        self.start_time = 0
        self.end_time   = 0

    def get_payload(self, prompt):
        return {
            "model": MODEL_NAME_TEST,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": MAX_TOKENS,
            "temperature": 0.7,
        }

    def get_token_count(self, data):
        try:
            return data["usage"]["completion_tokens"]
        except Exception:
            return 0

    async def worker(self, session, semaphore, req_id):
        prompt = random.choice(PROMPTS)
        t0 = time.time()
        headers = {"Content-Type": "application/json", "User-Agent": "stress-test/2.0"}
        if API_KEY:
            headers["Authorization"] = f"Bearer {API_KEY}"

        async with semaphore:
            last_err = None
            for attempt in range(MAX_RETRIES + 1):
                try:
                    async with session.post(
                        API_URL, json=self.get_payload(prompt), headers=headers
                    ) as resp:
                        latency = time.time() - t0
                        body    = await resp.text()
                        if resp.status == 200:
                            data   = _json.loads(body)
                            tokens = self.get_token_count(data)
                            self.results.append({"id": req_id, "status": "success",
                                "latency": latency, "tokens": tokens, "timestamp": time.time()})
                            return
                        if resp.status in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
                            await asyncio.sleep(2 ** attempt + random.uniform(0, 0.5))
                            last_err = f"{resp.status}: {body[:200]}"
                            continue
                        self.results.append({"id": req_id, "status": "error",
                            "latency": latency, "tokens": 0,
                            "error_msg": f"{resp.status}: {body[:200]}", "timestamp": time.time()})
                        return
                except Exception as exc:
                    last_err = str(exc)
                    if attempt < MAX_RETRIES:
                        await asyncio.sleep(2 ** attempt + random.uniform(0, 0.5))
                    else:
                        self.results.append({"id": req_id, "status": "exception",
                            "latency": time.time() - t0, "tokens": 0,
                            "error_msg": last_err, "timestamp": time.time()})

    async def run(self):
        print(f"🔥 STARTING STRESS TEST")
        print(f"   Requests    : {TOTAL_REQUESTS}")
        print(f"   Concurrency : {CONCURRENCY_LIMIT}")
        print(f"   Max-tokens  : {MAX_TOKENS}")
        print(f"   Retries     : {MAX_RETRIES}")
        print(f"   Target      : {API_URL}")
        print(f"   Model       : {MODEL_NAME_TEST}")
        print()
        self.start_time = time.time()
        sem       = asyncio.Semaphore(CONCURRENCY_LIMIT)
        timeout   = aiohttp.ClientTimeout(total=180)
        connector = aiohttp.TCPConnector(limit=CONCURRENCY_LIMIT, limit_per_host=CONCURRENCY_LIMIT)
        async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
            tasks = [self.worker(session, sem, i) for i in range(TOTAL_REQUESTS)]
            done, step = 0, max(1, TOTAL_REQUESTS // 10)
            for fut in asyncio.as_completed(tasks):
                await fut
                done += 1
                if done % step == 0:
                    ok = sum(1 for r in self.results if r["status"] == "success")
                    print(f"   Progress: {done:>4}/{TOTAL_REQUESTS}  ({done/TOTAL_REQUESTS*100:3.0f}%)  ✅ {ok}")
        self.end_time = time.time()
        print("\n✅ TEST COMPLETE.")

    def analyze(self):
        df       = pd.DataFrame(self.results)
        ok       = df[df["status"] == "success"].copy()
        duration = self.end_time - self.start_time
        total_tokens = int(ok["tokens"].sum()) if not ok.empty else 0
        rps          = len(ok) / duration if duration else 0
        tps          = total_tokens / duration if duration else 0
        avg_lat      = ok["latency"].mean() if not ok.empty else 0
        fail_rate    = (len(df) - len(ok)) / len(df) * 100 if len(df) else 0
        lats = ok["latency"].values if not ok.empty else np.array([0])
        p50, p95, p99 = np.percentile(lats, [50, 95, 99])
        print("\n" + "=" * 45)
        print("📊  PERFORMANCE METRICS")
        print("=" * 45)
        print(f"⏱️   Duration          : {duration:.2f} s")
        print(f"📨  Total requests    : {len(df)}")
        print(f"✅  Successful        : {len(ok)}")
        print(f"❌  Failed            : {len(df) - len(ok)}  ({fail_rate:.2f}%)")
        print(f"🚀  Throughput (RPS)  : {rps:.2f} req/s")
        print(f"⚡  Token gen speed   : {tps:.2f} tok/s")
        print(f"⏳  Avg latency       : {avg_lat:.3f} s")
        print(f"📈  P50 latency       : {p50:.3f} s")
        print(f"📈  P95 latency       : {p95:.3f} s")
        print(f"📈  P99 latency       : {p99:.3f} s")
        print("=" * 45)
        errs = df[df["status"] != "success"]
        if not errs.empty:
            print("\n🚨  FIRST ERROR SAMPLE:")
            print(errs.iloc[0].get("error_msg", "unknown"))
        if not ok.empty:
            self._plot(ok, df)

    def _plot(self, ok, full_df):
        plt.style.use("bmh")
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
        ax1.hist(ok["latency"], bins=30, color="#3498db", edgecolor="white")
        for pct, col, lbl in [(50,"#2ecc71","P50"),(95,"#f39c12","P95"),(99,"#e74c3c","P99")]:
            v = np.percentile(ok["latency"], pct)
            ax1.axvline(v, color=col, linestyle="--", linewidth=1.5, label=f"{lbl} {v:.2f}s")
        ax1.set_title("Latency Distribution"); ax1.set_xlabel("Seconds")
        ax1.set_ylabel("Count"); ax1.legend(fontsize=8)
        ok = ok.copy()
        ok["datetime"] = pd.to_datetime(ok["timestamp"], unit="s")
        ts = ok.set_index("datetime").resample("1s").count()["id"]
        ax2.plot(ts.index, ts.values, color="#2ecc71", linewidth=2)
        ax2.fill_between(ts.index, ts.values, alpha=0.2, color="#2ecc71")
        ax2.set_title("Successful Requests / Second"); ax2.set_xlabel("Time"); ax2.set_ylabel("req/s")
        counts = full_df["status"].value_counts()
        cmap   = {"success":"#2ecc71","error":"#e74c3c","exception":"#f1c40f"}
        ax3.pie(counts, labels=counts.index, autopct="%1.1f%%",
                colors=[cmap.get(s,"#95a5a6") for s in counts.index],
                startangle=90, wedgeprops=dict(edgecolor="white"))
        ax3.set_title("Success vs Failure Rate")
        plt.tight_layout(); plt.show()

tester = StressTester()
await tester.run()
tester.analyze()


## Usage from outside Kaggle

Replace `PUBLIC_URL` with the URL printed by Cell 5.

### curl
```bash
curl -s -X POST PUBLIC_URL/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -d '{"model":"Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4","messages":[{"role":"user","content":"Hello!"}],"max_tokens":200}' \\
  | python -m json.tool
```

### Python openai SDK
```python
from openai import OpenAI
client = OpenAI(base_url="PUBLIC_URL/v1", api_key="my-key")
resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    messages=[{"role":"user","content":"Explain LoRA in two sentences."}],
    max_tokens=300,
)
print(resp.choices[0].message.content)
```

### Stop server + tunnel
```python
vllm_proc.terminate()
cf_proc.terminate()
```
